# 구조화 출력 프롬프트 진단

같은 JSON 요구를 세 가지 프롬프트로 표현했을 때 파싱과 필드 존재 여부는 어떻게 달라질까요?

## 패키지 준비

첫 실행에서 수업 기준 버전을 설치합니다.

In [1]:
# 현재 커널에 수업 기준 패키지 버전을 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## API 키와 모델 준비

상위 폴더의 `.env`를 찾고 키의 존재만 확인합니다.

In [2]:
# find_dotenv는 실행 폴더부터 상위 폴더의 .env를 찾고 키 값은 출력하지 않습니다.
import json, os, time
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 루트의 .env에 OPENAI_API_KEY가 필요합니다.")
model = init_chat_model("openai:gpt-5.4-mini", max_tokens=220)

## 고정 과제

모든 절에서 같은 고객 문의와 확인 단어를 사용합니다.

In [3]:
# TASK는 모든 비교에서 바꾸지 않는 합성 문의이고 MENTIONS는 품질 점수가 아닙니다.
TASK = """고객 문의: 어제 받은 텀블러 뚜껑이 깨졌습니다. 주문번호는 A-1042입니다.
새 상품으로 바꾸고 싶습니다. 회사 규정상 교환 접수에는 파손 사진이 필요합니다.
답변에는 분류, 필요한 다음 행동, 고객에게 보낼 한 문장을 포함하세요."""
MENTIONS = ["교환", "A-1042", "사진"]

## 측정 함수

단어 포함 표시는 원문 정독을 돕는 표시이며 품질 점수가 아닙니다.

`run(messages)`는 이 노트북에서 작성하는 보조 함수입니다. 역할·내용 목록을 받아 `model.invoke()`를 한 번 실행하고, 답 문자열·사용량·경과 초를 담은 딕셔너리를 반환합니다. `def` 셀은 함수를 정의할 뿐이며 뒤에서 `run(...)`을 호출할 때 모델 요청이 시작됩니다. `mention_flags(text)`는 지정 단어가 문자열에 있는지 표시한 딕셔너리를 반환합니다. 답의 뜻을 판단하는 모델 호출은 포함하지 않습니다.

In [4]:
# 모델 응답의 원문, 사용량, 경과 시간을 한 레코드로 보존합니다.
def run(messages):
    started = time.perf_counter()
    answer = model.invoke(messages)
    return {"text": answer.text, "usage": answer.usage_metadata,
            "latency_s": round(time.perf_counter() - started, 3)}

def mention_flags(text):
    return {word: word in text for word in MENTIONS}

## 세 프롬프트 조건 준비

질문과 필요한 필드는 같고 JSON 지시의 구체성만 다릅니다.

In [5]:
# 질문은 고정하고 JSON 지시의 구체성만 세 단계로 바꿉니다.
prompts = {
    "brief": "JSON으로 답하세요. " + TASK,
    "fields": "JSON 객체로 답하세요. 필드는 category, next_action, customer_message입니다. " + TASK,
    "schema": "설명이나 코드 울타리 없이 JSON 객체 하나만 답하세요. "
              "category는 배송/교환/환불 중 하나이고 모든 필드는 문자열입니다. "
              "필드: category, next_action, customer_message. " + TASK,
}
results = {name: run([("system", "고객 지원 데이터를 만듭니다."), ("user", text)])
           for name, text in prompts.items()}

## 실제 파싱과 필드 검사

파싱 실패를 가정하지 않고 각 원문에 `json.loads`를 실제 적용합니다.

`json.loads()`의 입력은 응답 문자열이고 반환값은 JSON 내용에 대응하는 Python 값입니다. JSON 객체는 딕셔너리, 배열은 목록이 됩니다. 파싱 성공만으로 원하는 객체나 필드를 받았다고 볼 수 없습니다. 이번에 저장된 세 응답은 객체이지만 일반 입력에 대한 자료형 검증까지 이 코드가 수행하는 것은 아닙니다.

`{f: f in parsed for f in fields}`는 필요한 필드별로 키가 있는지 표시하는 사전 표현식입니다. 키 값의 타입·허용값·빈 문자열·업무 사실은 검사하지 않습니다. 예를 들어 빈 category도 키 자체가 있으면 이 검사에서는 참입니다.

일정 자료의 “일시”를 프로그램이 기대하는 `starts_at`으로 바꾸는 문제와 날짜가 실제로 맞는지 확인하는 문제도 다릅니다. 아래 결과에서는 파싱 성공과 영문 필드 충족부터 따로 읽고, 전체 검증이 끝났다고 해석하지 않습니다.

In [6]:
# json.loads 파싱 성공과 필요한 영문 필드 존재를 별도로 기록합니다.
diagnostics = {}
for name, result in results.items():
    try:
        parsed = json.loads(result["text"])
        fields = ["category", "next_action", "customer_message"]
        diagnostics[name] = {"parsed": True, "fields": {f: f in parsed for f in fields}}
    except (json.JSONDecodeError, TypeError) as error:
        diagnostics[name] = {"parsed": False, "error": type(error).__name__}
print(json.dumps(diagnostics, ensure_ascii=False, indent=2))

{
  "brief": {
    "parsed": true,
    "fields": {
      "category": false,
      "next_action": false,
      "customer_message": false
    }
  },
  "fields": {
    "parsed": true,
    "fields": {
      "category": true,
      "next_action": true,
      "customer_message": true
    }
  },
  "schema": {
    "parsed": true,
    "fields": {
      "category": true,
      "next_action": true,
      "customer_message": true
    }
  }
}


## 원문·사용량·진단 함께 보기

모두 성공했다면 이번 표본에서 파싱 실패를 관찰하지 못했다고 기록합니다.

In [7]:
# 각 조건의 raw 응답을 생략하지 않고 단어 표시와 사용량 옆에 출력합니다.
for name, result in results.items():
    print("\n", name, diagnostics[name], result["usage"], result["latency_s"])
    print(result["text"])


 brief {'parsed': True, 'fields': {'category': False, 'next_action': False, 'customer_message': False}} {'input_tokens': 98, 'output_tokens': 146, 'total_tokens': 244, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.38
{
  "분류": "상품 파손으로 인한 교환 요청",
  "필요한 다음 행동": [
    "파손된 텀블러 뚜껑의 사진을 첨부해 주세요.",
    "주문번호 A-1042를 확인한 뒤 교환 접수를 진행하겠습니다."
  ],
  "고객에게 보낼 한 문장": "어제 받으신 텀블러 뚜껑 파손으로 불편을 드려 죄송합니다. 주문번호 A-1042의 교환 접수를 위해 파손 사진을 보내주시면 바로 새 상품 교환을 도와드리겠습니다."
}

 fields {'parsed': True, 'fields': {'category': True, 'next_action': True, 'customer_message': True}} {'input_tokens': 110, 'output_tokens': 89, 'total_tokens': 199, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.199
{"category":"교환/파손","next_action":"고객에게 파손된 뚜껑의 사진 제출을 요청하고, 주문번호 A-1042로 교환 접수를 진행한다.","customer_message":"주문번호 A-1042 확인했습니다. 파손된 텀블러 뚜껑의 사진을 보내주시면 교환 접수를 도와드리겠습니다."}

 schema {'parsed': True, 'fie